<a href="https://colab.research.google.com/github/Prachetaganguly/pytorch-for-neural-networks/blob/main/Pytorch_for_CNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [117]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.utils import make_grid

import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
%matplotlib inline

In [118]:
#convert MIST image files into tensor of 4-D( no of images, ht, wdth, colour channel)
transform=transforms.ToTensor()

In [119]:
#Train Data
train_data=datasets.MNIST(root='data',train=True,download=True,transform=transform)

In [120]:
#Test Data
test_data=datasets.MNIST(root='data',train=False,download=True,transform=transform)

In [121]:
train_data
#tarin data has 60000 data points

Dataset MNIST
    Number of datapoints: 60000
    Root location: data
    Split: Train
    StandardTransform
Transform: ToTensor()

In [122]:
test_data
#test set has 10000 data points

Dataset MNIST
    Number of datapoints: 10000
    Root location: data
    Split: Test
    StandardTransform
Transform: ToTensor()

In [123]:
#Create a small batch size for images...let's say 10
train_loader=DataLoader(train_data, batch_size=10, shuffle=True)
test_loader=DataLoader(test_data, batch_size=10, shuffle=False)

In [124]:
#Define our CNN model
#Describe convolutional layer and what it's doing(2 layers)
conv1=nn.Conv2d(1,6,3,1)
conv2=nn.Conv2d(6,16,3,1)

In [125]:
#Grab 1 MNIST record/image
for i, (X_Train, y_train) in enumerate(train_data):
    break

In [126]:
X_Train.shape

torch.Size([1, 28, 28])

In [127]:
x=X_Train.view(1,1,28,28) #Convert it into 4-D

In [128]:
x=F.relu(conv1(x))

In [129]:
x.shape

torch.Size([1, 6, 26, 26])

In [130]:
#pass thru pooling layer
x=F.max_pool2d(x,2,2) #kernel of 2x2 and stride of 2

In [131]:
x.shape

torch.Size([1, 6, 13, 13])

In [132]:
#2nd convolutional layer
x=F.relu(conv2(x))

In [133]:
x.shape #again we didn't set padding so we lose 2 pixels around the outside of the image

torch.Size([1, 16, 11, 11])

In [134]:
#pooling layer
x=F.max_pool2d(x,2,2)

In [135]:
x.shape #11/2=5.5 but we have to round down, bcoz u can't invent data to round up

torch.Size([1, 16, 5, 5])

In [136]:
#Model Class
class ConvolutionalNetwork(nn.Module):
  def __init__(self):
    super().__init__() # Call the superclass constructor
    self.conv1=nn.Conv2d(1,6,3,1)
    self.conv2=nn.Conv2d(6,16,3,1)
    #Fully Connected Layer
    self.fc1=nn.Linear(5*5*16,120)
    self.fc2=nn.Linear(120,84)
    self.fc3=nn.Linear(84,10)

  def forward(self,X):
    X=F.relu(self.conv1(X))
    X=F.max_pool2d(X,2,2)

    #Re-view to flatten it out
    X=X.view(-1, 16*5*5) #negative one so that we can vary the batch size

    #Fully Connected Layers
    X=F.relu(self.fc1(X))
    X=F.relu(self.fc2(X))
    X=self.fc3(X) # Fixed: Changed x to X for consistency
    return F.log_softmax(X, dim=1)

In [137]:
#Create an Instance of our model
torch.manual_seed(41)
model=ConvolutionalNetwork()
model

ConvolutionalNetwork(
  (conv1): Conv2d(1, 6, kernel_size=(3, 3), stride=(1, 1))
  (conv2): Conv2d(6, 16, kernel_size=(3, 3), stride=(1, 1))
  (fc1): Linear(in_features=400, out_features=120, bias=True)
  (fc2): Linear(in_features=120, out_features=84, bias=True)
  (fc3): Linear(in_features=84, out_features=10, bias=True)
)

In [138]:
#Loss Function Optimizer
criterion=nn.CrossEntropyLoss()
torch.optim.Adam(model.parameters(), lr=0.01)

Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.01
    maximize: False
    weight_decay: 0
)